## SEINE Video Generation model

In [1]:
import os
import sys
import math
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'SEINE'))

import utils
from diffusion import create_diffusion

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import argparse
import torchvision

from einops import rearrange
from models import get_models
from torchvision.utils import save_image
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from omegaconf import OmegaConf
from PIL import Image
import numpy as np
from torchvision import transforms
from SEINE import video_transforms
# from dataset import video_transforms
from utils import mask_generation_before
from natsort import natsorted
from diffusers.utils.import_utils import is_xformers_available
import pdb
import datetime

class SeineModel:
    def __init__(self, args):
        print('Initializing SEINE model...')

        if args.seed:
            torch.manual_seed(args.seed)
        torch.set_grad_enabled(False)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if args.ckpt is None:
            raise ValueError("Please specify a checkpoint path using --ckpt <path>")

        # load model
        self.latent_h = args.image_size[0] // 8
        self.latent_w = args.image_size[1] // 8
        self.image_h = args.image_size[0]
        self.image_w = args.image_size[1]
        self.model = get_models(args).to(self.device)

        if args.enable_xformers_memory_efficient_attention:
            if is_xformers_available():
                self.model.enable_xformers_memory_efficient_attention()
            else:
                raise ValueError("xformers is not available. Make sure it is installed correctly")

        ckpt_path = args.ckpt 
        state_dict = torch.load(ckpt_path, map_location=lambda storage, loc: storage)['ema']
        self.model.load_state_dict(state_dict)

        self.model.eval()
        pretrained_model_path = args.pretrained_model_path
        self.diffusion = create_diffusion(str(args.num_sampling_steps))
        self.vae = AutoencoderKL.from_pretrained(pretrained_model_path, subfolder="vae").to(self.device)
        self.text_encoder = TextEmbedder(pretrained_model_path).to(self.device)
        if args.use_fp16:
            # print('Warning: using half percision for inferencing!')
            self.vae.to(dtype=torch.float16)
            self.model.to(dtype=torch.float16)
            self.text_encoder.to(dtype=torch.float16)

        self.mask_type = args.mask_type
        self.num_frames = args.num_frames
        self.use_fp16 = args.use_fp16
        self.do_classifier_free_guidance = args.do_classifier_free_guidance
        self.sample_method = args.sample_method
        self.cfg_scale = args.cfg_scale
        self.use_mask = args.use_mask

        print('Initialization complete!')

    def get_input(self, input_path):
        transform_video = transforms.Compose([
                            video_transforms.ToTensorVideo(), # TCHW
                            video_transforms.ResizeVideo((self.image_h, self.image_w)),
                            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
                        ])
        if input_path is not None:
            print(f'Loading video from {input_path}...')
            if os.path.isdir(input_path):
                file_list = os.listdir(input_path)
                video_frames = []
                if self.mask_type.startswith('onelast'):
                    num = int(self.mask_type.split('onelast')[-1])
                    # get first and last frame
                    first_frame_path = os.path.join(input_path, natsorted(file_list)[0])
                    last_frame_path = os.path.join(input_path, natsorted(file_list)[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(first_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    last_frame = torch.as_tensor(np.array(Image.open(last_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    # add zeros to frames
                    num_zeros = self.num_frames-2*num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    for i in range(num):
                        video_frames.append(last_frame)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                else:
                    for file in file_list:
                        if file.endswith('jpg') or file.endswith('png'):
                            image = torch.as_tensor(np.array(Image.open(file), dtype=np.uint8, copy=True)).unsqueeze(0)
                            video_frames.append(image)
                        else:
                            continue
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                return video_frames, n
            elif os.path.isfile(input_path):
                _, full_file_name = os.path.split(input_path)
                file_name, extension = os.path.splitext(full_file_name)
                if extension == '.jpg' or extension == '.png':
                    print("Loading the input image...")
                    video_frames = []
                    num = int(self.mask_type.split('first')[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(input_path).convert('RGB'), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    num_zeros = self.num_frames-num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                    return video_frames, n
                else:
                    raise TypeError(f'{extension} is not supported !!')
            else:
                raise ValueError('Please check your path input!!')
        else:
            raise ValueError('Need to give a video or some images')

    def auto_inpainting(self, video_input, masked_video, mask, prompt, negative_prompt):
        b,f,c,h,w = video_input.shape

        # prepare inputs
        if self.use_fp16:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, dtype=torch.float16, device=self.device) # b,c,f,h,w
            masked_video = masked_video.to(dtype=torch.float16)
            mask = mask.to(dtype=torch.float16)
        else:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, device=self.device) # b,c,f,h,w

        masked_video = rearrange(masked_video, 'b f c h w -> (b f) c h w').contiguous()
        masked_video = self.vae.encode(masked_video).latent_dist.sample().mul_(0.18215)
        masked_video = rearrange(masked_video, '(b f) c h w -> b c f h w', b=b).contiguous()
        mask = torch.nn.functional.interpolate(mask[:,:,0,:], size=(self.latent_h, self.latent_w)).unsqueeze(1)
    
        # classifier_free_guidance
        if self.do_classifier_free_guidance:
            masked_video = torch.cat([masked_video] * 2)
            mask = torch.cat([mask] * 2)
            z = torch.cat([z] * 2)
            prompt_all = [prompt] + [negative_prompt]

        else:
            masked_video = masked_video
            mask = mask
            z = z
            prompt_all = [prompt]

        text_prompt = self.text_encoder(text_prompts=prompt_all, train=False)
        model_kwargs = dict(encoder_hidden_states=text_prompt, 
                                class_labels=None, 
                                cfg_scale=self.cfg_scale,
                                use_fp16=self.use_fp16,) # tav unet

        # sample video
        if self.sample_method == 'ddim':
            samples = self.diffusion.ddim_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        elif self.sample_method == 'ddpm':
            samples = self.diffusion.p_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        samples, _ = samples.chunk(2, dim=0) # [1, 4, 16, 32, 32]
        if self.use_fp16:
            samples = samples.to(dtype=torch.float16)

        video_clip = samples[0].permute(1, 0, 2, 3).contiguous() # [16, 4, 32, 32]
        video_clip = self.vae.decode(video_clip / 0.18215).sample # [16, 3, 256, 256]

        return video_clip

    def generate_video(self, args, save_path):
        prompt = args.text_prompt
        
        if prompt == []:
            prompt = args.input_path.split('/')[-1].split('.')[0].replace('_', ' ')
        else:
            prompt = prompt[0]
        prompt_base = prompt.replace(' ','_')

        if not os.path.exists(os.path.join(save_path)):
            os.makedirs(os.path.join(save_path))
        video_input, reserve_frames = self.get_input(args.input_path) # f,c,h,w
        video_input = video_input.to(self.device).unsqueeze(0)  # b,f,c,h,w
        mask = mask_generation_before(self.mask_type, video_input.shape, video_input.dtype, self.device) # b,f,c,h,w
        masked_video = video_input * (mask == 0)

        video_clip = self.auto_inpainting(video_input, masked_video, mask, prompt, args.negative_prompt)
        video_ = ((video_clip * 0.5 + 0.5) * 255).add_(0.5).clamp_(0, 255).to(dtype=torch.uint8).cpu().permute(0, 2, 3, 1)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        # filename = f"{prompt_base}_{timestamp}.mp4"
        filename = f"{timestamp}.mp4"
        save_video_path = os.path.join(save_path, filename)
        torchvision.io.write_video(save_video_path, video_, fps=8)
        print(f'Video saved in {save_video_path}')

        return save_video_path

/home/brina/miniconda3/envs/seine/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load SEINE model with configs

In [2]:
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="./configs/seine.yaml")
args, unknown = parser.parse_known_args()
omega_conf = OmegaConf.load(args.config)
seine_model = SeineModel(omega_conf)

Initializing SEINE model...
Initialization complete!


## PromptPilot Agent

In [3]:
from openai import OpenAI
import yaml
import re
from argparse import Namespace
import base64
from PIL import Image
from typing import List, Dict
from transformers import (
    CLIPProcessor, CLIPModel
)
import torchvision.transforms as T
import cv2
import numpy as np
import json

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class PromptPilotAgent:
    def __init__(self, seine_model, device="cuda"):
        self.llm = OpenAI(
                api_key="fy3jHNMV7OC7t7fQprQkFgp7NeSlRsMG",
                base_url="https://api.deepinfra.com/v1/openai",
            )
        self.temperature = 0.0
        self.max_iterations = 2
        self.seine_model = seine_model
        self.device = device

        self.SYSTEM_PROMPT = """
                                You are a prompt refinement agent specialized in enhancing prompts for video generation models that take a single image as input.
                                Your goal is to improve the given prompt so that the generated video is visually coherent, smooth in motion, and logically consistent with the content and context of the image.
                                Carefully consider the visual elements and implied actions in the image, and rewrite the prompt to guide the model toward generating a realistic and temporally logical video sequence.
                             """

        self.USER_PROMPT = """
                              Given an image and a history of previous prompts with their corresponding scores for their generated videos, refine and rewrite the prompt to  further enhance the video quality. Ensure that the refined prompt is an improved version of the previous prompts, providing more guidance as appropriate.

                              Output the refined prompt in the following format:
                              Refined prompt: <refined prompt>

                              If you are not able to refine the prompt, output the following:
                              Refined prompt: <previous prompt>

                           """
        # Given an image and the previous prompt, refine the prompt and also create a negative prompt to result in a better video. Output the refined prompt and negative prompt in the following format:
        #                       Refined prompt: <refined prompt>
        #                       Negative prompt: <negative prompt>
                            #   For example,
                            #   Previous prompt: cat walking around
                            #   CLIP Alignment Score: 0.271
                            #   Temporal Consistency Score: 13.25
                            #   Dynamic Degree Score: 4.38
                            #   Refined prompt: 

        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

        self.resize = T.Resize((224, 224))
        self.to_tensor = T.ToTensor()
        self.scores = {
            "clip_tva_score": 0,
            "temporal_consistency": 0,
            "dynamic_degree": 0}

    def extract_frames(self,video_path: str, num_frames: int = 4) -> List[Image.Image]:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

        frames = []
        for idx in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(rgb_frame))
        cap.release()
        return frames

    def compute_clip_alignment(self, frames: List[Image.Image], prompt: str) -> float:
        # runcate prompt to CLIP's max token limit (77)
        truncated_prompt = clip_processor.tokenizer.decode(
            clip_processor.tokenizer(prompt, truncation=True, max_length=77)["input_ids"],
            skip_special_tokens=True
        )
        inputs = clip_processor(
            text=[truncated_prompt] * len(frames), images=frames,
            return_tensors="pt", padding=True
        ).to(self.device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
            sims = torch.cosine_similarity(outputs.image_embeds, outputs.text_embeds)
        return sims.mean().item()
    
    def compute_temporal_consistency(self, frames: List[Image.Image]) -> float:
        gray_frames = [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2GRAY) for f in frames]
        total_flow = 0.0
        for i in range(1, len(gray_frames)):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i - 1], gray_frames[i], None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0
            )
            magnitude = np.linalg.norm(flow, axis=2).mean()
            total_flow += magnitude
        return total_flow / (len(frames) - 1)

    def compute_dynamic_degree(self, frames: List[Image.Image]) -> float:
        """
        Computes dynamic degree as the variance of frame-to-frame pixel differences.
        Higher values imply more movement or dynamic content.
        """
        gray_frames = [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2GRAY) for f in frames]
        diffs = []

        for i in range(1, len(gray_frames)):
            diff = np.abs(gray_frames[i].astype(np.float32) - gray_frames[i - 1].astype(np.float32))
            mean_diff = diff.mean()
            diffs.append(mean_diff)

        return np.var(diffs) if diffs else 0.0

    def evaluate_video(self, frames: List[Image.Image], prompt: str) -> Dict[str, float]:
        # print("frames",frames)
        clip_score = self.compute_clip_alignment(frames, prompt)
        tc_score = self.compute_temporal_consistency(frames)
        dd_score = self.compute_dynamic_degree(frames)

        self.scores.update({
            "clip_tva_score": clip_score,
            "temporal_consistency": tc_score,
            "dynamic_degree": dd_score
        })

        return self.scores  # Return updated dictionary if needed
    
    def set_first_text_prompt(self, yaml_path):
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # Extract the input_path value to use as the new_prompt
        input_path = content.get("input_path", "No input_path found")
        prompt = input_path.split('/')[-1].split('.')[0].replace('_', ' ')
    
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: ["{prompt}"]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def update_text_prompt(self, yaml_path, refined_prompt):
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: ["{refined_prompt}"]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def query_llm(self):
        response = self.llm.chat.completions.create(
            # model="meta-llama/Llama-3.2-11B-Vision-Instruct",
            model="meta-llama/Llama-3.2-90B-Vision-Instruct",
            messages=self.messages,
            temperature=self.temperature
        )
        return response.choices[0].message.content

    def clean_response(self, response):
        # refined_match = re.search(r"Refined prompt:\s*((?:.|\n)*?)\Z", response)
        # refined_prompt = refined_match.group(1).strip().replace('\n', ' ') if refined_match else ""

        refined_match = re.search(r"Refined prompt:\s*(.*?\.)(\s|$)", response)

        refined_prompt = refined_match.group(1).strip() if refined_match else ""

        # print("Refined:", refined_prompt)

        return refined_prompt
    
    def cleanup(self):
        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]
        torch.cuda.empty_cache()

    def run(self, yaml_path):
        print("Generating video...")

        # set text prompt for the first time
        self.set_first_text_prompt(yaml_path)

        # Read the file content
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # get the string inside the text_prompt list
        # prompt = content.get("text_prompt", [None])[0]
        input_path = content.get("input_path", None)
        save_path = content.get("save_path", None)
        save_path = save_path.replace("./results/", "./results/exp3/")

        # prepare llm image prompt
        with open(input_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

        self.messages.append(
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64, {encoded_image}"
                            }
                        }
                    ]
                }
            )

        clip_score = None
        tc_score = None
        dd_score = None
        history = ""

        for iteration in range(self.max_iterations):
            with open(yaml_path, 'r') as f:
                content = yaml.safe_load(f)

            # get the string inside the text_prompt list
            current_prompt = content.get("text_prompt", [None])[0]

            print(f"\nIteration {iteration+1}/{self.max_iterations}")
            print("Current prompt:", current_prompt)

            # prepare llm text prompt
            if history == "":
                history += f"""
                                Previous prompt: {current_prompt}
                            """

                text = self.USER_PROMPT + \
                        f"""
                            {history}

                            Refined prompt:
                        """

            else:
                history += f"""
                                Previous prompt: {current_prompt}
                                CLIP Alignment score: {clip_score}
                                Temporal Consistency score: {tc_score}
                                Dynamic Degree score: {dd_score}
                            """
                
                text = f"""
                            CLIP Alignment Score measures how well the generated video frames align with the given text prompt, so a higher score means better semantic alignment with the prompt. Temporal Consistency Score assesses the smoothness and coherence between consecutive video frames, so a higher score means fewer visual artifacts or jumps, meaning the video flows naturally over time. Dynamic Degree Score quantifies the amount of motion in the video by comparing pixel changes across frames, so a higher score means more dynamic movement (good for action prompts, not ideal for static scenes).
                        """ + \
                        self.USER_PROMPT + \
                        f"""
                            {history}

                            Refined prompt:
                        """

            # create llm use prompt
            self.messages.append(
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": text
                        }
                    ]
                }
            )

            # get llm response
            response = self.query_llm()
            print(response)

            # clean up response
            refined_prompt = self.clean_response(response)

            if refined_prompt == current_prompt:
                break

            # update yaml
            self.update_text_prompt(yaml_path, refined_prompt)

            # prepare args for video generation
            with open(yaml_path, "r") as f:
                content = yaml.safe_load(f)
            args = Namespace(**content)
            video_path = self.seine_model.generate_video(args, save_path)

            # compute evaluation metrics
            frames = self.extract_frames(video_path)
            scores = self.evaluate_video(frames, refined_prompt)
            # print(f"Prompt: {refined_prompt}")
            print(f"Scores: {scores}")

            clip_score = scores["clip_tva_score"]
            tc_score = scores["temporal_consistency"]
            dd_score = scores["dynamic_degree"]

            # Save to JSON
            result = {
                "prompt": refined_prompt,
                "scores": {k: float(v) for k, v in scores.items()}
            }
            json_path = f"{save_path}/scores.json"

            if os.path.exists(json_path):
                with open(json_path, "r") as f:
                    data = json.load(f)
            else:
                data = []

            data.append(result)

            with open(json_path, "w") as f:
                json.dump(data, f, indent=4)

        # cleanup after generation
        self.cleanup()

## Create Prompt Agent

In [4]:
agent = PromptPilotAgent(seine_model)

directory = "configs/images"

# Using os.walk to loop through subdirectories
yaml_paths = []
for subdir, _, files in os.walk(directory):
    for file in files:
        if file.endswith('.yaml'):
            # Add the full path of each YAML file
            yaml_paths.append(os.path.join(subdir, file))

# Process each YAML file
for yaml_path in yaml_paths:
    print(f"\nProcessing: {yaml_path}")
    agent.run(yaml_path)


Processing: configs/images/Animals/penguin_walking.yaml
Generating video...

Iteration 1/2
Current prompt: penguin walking
Refined prompt: A penguin walking on a rocky terrain, with its flippers swaying and its feet waddling in a smooth, natural motion, as if it is moving towards a specific destination, with the surrounding environment and lighting changing accordingly to create a sense of depth and realism.
Loading video from dataset/Animals/penguin_walking.png...
Loading the input image...


100%|██████████| 250/250 [03:20<00:00,  1.24it/s]


Video saved in ./results/exp3/Animals/penguin_walking/20250426_123800.mp4
Scores: {'clip_tva_score': 0.3221805691719055, 'temporal_consistency': 1.4431988795598347, 'dynamic_degree': 3.0893962}

Iteration 2/2
Current prompt: A penguin walking on a rocky terrain, with its flippers swaying and its feet waddling in a smooth, natural motion, as if it is moving towards a specific destination, with the surrounding environment and lighting changing accordingly to create a sense of depth and realism.
Refined prompt: A penguin walking on a rocky terrain, with its flippers swaying and its feet waddling in a smooth, natural motion, as if it is moving towards a specific destination, with the surrounding environment and lighting changing accordingly to create a sense of depth and realism, and the penguin's movements are fluid and synchronized with the changing scenery, creating a visually appealing and immersive video sequence.
Loading video from dataset/Animals/penguin_walking.png...
Loading the i

100%|██████████| 250/250 [03:33<00:00,  1.17it/s]


Video saved in ./results/exp3/Animals/penguin_walking/20250426_124144.mp4
Scores: {'clip_tva_score': 0.3320733308792114, 'temporal_consistency': 1.3797459801038106, 'dynamic_degree': 30.158606}

Processing: configs/images/Animals/herd_of_elephants_walking.yaml
Generating video...

Iteration 1/2
Current prompt: herd of elephants walking
Refined prompt: A herd of elephants walking in a savannah, with the elephants moving in a coordinated manner, their trunks and tusks visible as they make their way across the grassy terrain.
Loading video from dataset/Animals/herd_of_elephants_walking.png...
Loading the input image...


...

Video saved in ./results/exp3/Outdoor/metropolitan_evening_view/20250427_131639.mp4
Scores: {'clip_tva_score': 0.30512624979019165, 'temporal_consistency': 3.0518277883529663, 'dynamic_degree': 8.903746}

Iteration 2/2
Current prompt: A metropolitan evening view with a vibrant cityscape, featuring a mix of modern skyscrapers and historic buildings, set against a backdrop of a stunning sunset with hues of orange, pink, and purple, and a bustling street scene with people walking and cars driving by, all captured in a single, cohesive, and visually appealing video sequence.
Refined prompt: A metropolitan evening view with a vibrant cityscape, featuring a mix of modern skyscrapers and historic buildings, set against a backdrop of a stunning sunset with hues of orange, pink, and purple, and a bustling street scene with people walking and cars driving by, all captured in a single, cohesive, and visually appealing video sequence, with a focus on smooth transitions between frames and a dynamic

100%|██████████| 250/250 [03:32<00:00,  1.18it/s]


Video saved in ./results/exp3/Outdoor/metropolitan_evening_view/20250427_132021.mp4
Scores: {'clip_tva_score': 0.33569133281707764, 'temporal_consistency': 6.09238600730896, 'dynamic_degree': 255.98955}

Processing: configs/images/Outdoor/huge_vessel_passing_ravine.yaml
Generating video...

Iteration 1/2
Current prompt: huge vessel passing ravine
Refined prompt: A large vessel navigates through a narrow ravine, showcasing its impressive size and maneuverability as it expertly navigates the challenging terrain.
Loading video from dataset/Outdoor/huge_vessel_passing_ravine.png...
Loading the input image...


100%|██████████| 250/250 [03:31<00:00,  1.18it/s]


Video saved in ./results/exp3/Outdoor/huge_vessel_passing_ravine/20250427_132401.mp4
Scores: {'clip_tva_score': 0.2646924555301666, 'temporal_consistency': 9.163166364034018, 'dynamic_degree': 13.728859}

Iteration 2/2
Current prompt: A large vessel navigates through a narrow ravine, showcasing its impressive size and maneuverability as it expertly navigates the challenging terrain.
Refined prompt: A large vessel navigates through a narrow ravine, showcasing its impressive size and maneuverability as it expertly navigates the challenging terrain.
